# 08 — Midterm report: every result, table, and figure

This notebook produces **all numbers, tables, and figures for the midterm report** from the finished runs.
Run it once from top to bottom on CARC. Nothing here trains a model or needs a GPU.

**What it reads**

| Source | What | Folder |
|---|---|---|
| Course-project runs | No replay, Random 10%, Random 14.3%, Lowest margin, MFR, Balanced MFR, At-Risk MFR | `group26/artifacts/runs` |
| Extended version of the code | DAPR-weak runs, plus any method missing above | `grp26-mfr-dpo/artifacts/runs` |
| Joint training (upper reference) | All three datasets trained together | `joint_runs/` in either folder |

**What it writes** (to `group26/artifacts/analysis/midterm/`, zipped at the end)

- `tables/*.csv`: every table, as numbers.
- `figures/*.png` (300 dpi) and `figures/*.pdf` (vector): every figure, ready to paste into the report.
- `key_numbers.txt`: every number quoted in the report text, in sentences.

**Where each output goes in the report:** see the last section, "Report map".

All results use the **validation split** (200 pairs per behavior) on the **8 development cells**
(orders 1–4 × seeds 0–1). The locked test set is not touched.

## 0. Settings

Edit only this cell. On CARC the defaults are correct.

- `METHODS`: methods to include, in display order. A method is shown only if all 8 cells exist
  (set `ALLOW_PARTIAL = True` to show incomplete methods in the descriptive tables too).
- `EXTENDED_FILL_METHODS`: methods that may be read from the extended folder **only when they are
  missing** from `group26`. Baselines (`none`, `random`, ...) always come from `group26`.

In [ ]:
import json, os, shutil, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from IPython.display import HTML, display

GROUP26_ARTIFACTS = Path(os.environ.get('MFR_OUTPUT_DIR', '/project2/xiangren_1987/group26/artifacts'))
EXTENDED_ARTIFACTS = Path(os.environ.get('MFR_EXTENDED_DIR', '/project2/xiangren_1987/grp26-mfr-dpo/artifacts'))

METHODS = ['none', 'random', 'random_high', 'mfr', 'mfr_balanced', 'mfr_at_risk',
           'mfr_stochastic', 'lowest_margin', 'dapr_weak']
EXTENDED_FILL_METHODS = ['dapr_weak', 'mfr_balanced', 'mfr_at_risk', 'mfr_stochastic']
EXTENSION_METHODS = ['dapr_weak']      # changes the training loss, not only the selection; flagged in tables
ORDERS = [1, 2, 3, 4]
SEEDS = [0, 1]
SPLIT = 'val'                          # never 'test' before the method is frozen
SCORE = 'accuracy'                     # primary metric; 'accuracy_sum' is the secondary one
MARGIN_COL = 'margin' if SCORE == 'accuracy' else 'margin_sum'
BASELINE = 'random'                    # all "vs" comparisons use equal-budget Random 10%
TOLERANCE = 2.0                        # final-task points a method may lose vs Random 10%
N_BOOT = 20000                         # bootstrap resamples over order-seed cells
BOOT_SEED = 544
ALLOW_PARTIAL = False
EXAMPLE_CELL = (1, 0)                  # (order, seed) used for the Safety example in the introduction
SAVE = True
OUT = GROUP26_ARTIFACTS / 'analysis' / 'midterm'

REPO = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / 'configs/experiment_protocol.json').exists()), None)
PROTOCOL = json.loads((REPO / 'configs/experiment_protocol.json').read_text()) if REPO else {}
if PROTOCOL:
    TOLERANCE = float(PROTOCOL.get('new_learning_tolerance_points', TOLERANCE))
    PROTOCOL_ORDERS = {int(k): list(v) for k, v in PROTOCOL['orders'].items()}
else:
    PROTOCOL_ORDERS = {}
print('group26 artifacts :', GROUP26_ARTIFACTS, '| exists:', GROUP26_ARTIFACTS.exists())
print('extended artifacts:', EXTENDED_ARTIFACTS, '| exists:', EXTENDED_ARTIFACTS.exists())
print('outputs           :', OUT if SAVE else 'not saved (SAVE = False)')

### Labels, colors, and plot style

Colors follow one rule: a method keeps the same color in every figure. Random 14.3% is a lighter
step of Random 10%'s blue because it is the same rule with more slots. Every bar and point also has a
text label, so no result depends on color alone.

In [ ]:
MLABEL = {'none': 'No replay', 'random': 'Random 10%', 'random_high': 'Random 14.3%',
          'lowest_margin': 'Lowest margin', 'mfr': 'MFR (ours)', 'mfr_balanced': 'Balanced MFR',
          'mfr_at_risk': 'At-Risk MFR', 'mfr_stochastic': 'Stochastic MFR', 'dapr_weak': 'DAPR-weak',
          'joint': 'Joint training'}
MCOLOR = {'none': '#8a8984', 'random': '#2a78d6', 'random_high': '#86b6ef', 'lowest_margin': '#1baf7a',
          'mfr': '#eb6834', 'mfr_balanced': '#eda100', 'mfr_at_risk': '#e87ba4', 'mfr_stochastic': '#e34948',
          'dapr_weak': '#6250d6', 'joint': '#0b0b0b'}
DATASETS = ['helpful', 'safe', 'quality']
DLABEL = {'helpful': 'Helpful', 'safe': 'Safe', 'quality': 'Quality'}
DCOLOR = {'helpful': '#2a78d6', 'safe': '#eb6834', 'quality': '#1baf7a'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e5e4e0'
FORGET_C, RECOVER_C, NEUTRAL = '#e34948', '#2a78d6', '#f0efec'
DIVERGING = LinearSegmentedColormap.from_list('div', [FORGET_C, NEUTRAL, RECOVER_C])
SEQUENTIAL = LinearSegmentedColormap.from_list('seq', ['#cde2fb', '#86b6ef', '#2a78d6', '#184f95', '#0d366b'])
KEY_METHODS = ['none', 'random', 'mfr', 'lowest_margin', 'dapr_weak']   # for line charts (fewer series)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8.5, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2,
    'ytick.color': INK2, 'text.color': INK, 'axes.grid': False, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'lines.linewidth': 2, 'legend.frameon': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
FULL_W, HALF_W = 6.5, 3.25   # inches: full text width of the report, and half of it

if SAVE:
    (OUT / 'tables').mkdir(parents=True, exist_ok=True)
    (OUT / 'figures').mkdir(parents=True, exist_ok=True)
display(HTML('''<style>.rt{border-collapse:collapse;font-size:12.5px;margin:6px 0 14px}
.rt th{background:#333;color:#fff;padding:5px 9px;text-align:left}
.rt td{padding:4px 9px;border-bottom:1px solid #ddd}</style>'''))

def show(df, name=None, caption=None, decimals=2):
    """Display a table and save it as CSV (and LaTeX when available)."""
    out = df.copy()
    nums = out.select_dtypes(include='number').columns
    out[nums] = out[nums].round(decimals)
    if caption:
        display(HTML(f'<b>{caption}</b>'))
    display(HTML(out.to_html(index=False, border=0, na_rep='—', classes='rt')))
    if SAVE and name:
        out.to_csv(OUT / 'tables' / f'{name}.csv', index=False)
        try:
            (OUT / 'tables' / f'{name}.tex').write_text(out.to_latex(index=False, na_rep='--', float_format='%.2f'))
        except Exception:
            pass

def save(fig, name):
    if SAVE:
        fig.savefig(OUT / 'figures' / f'{name}.png', bbox_inches='tight')
        fig.savefig(OUT / 'figures' / f'{name}.pdf', bbox_inches='tight')
    plt.show()

def clean_axis(ax, grid='x'):
    if grid:
        ax.grid(axis=grid, color=GRID, lw=0.8)
        ax.set_axisbelow(True)

## 1. Load the runs

A run is used only when its folder has `COMPLETE.json`, `settings.json`, and `results.csv`.
Methods other than No replay reuse the No replay run's Stage-1 model (`stage1_from`), so Stage 1 is
read from that run.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())

def scan_runs(root, source):
    found = []
    runs_dir = Path(root) / 'runs'
    if not runs_dir.exists():
        return found
    for folder in sorted(runs_dir.glob('*')):
        needed = [folder / 'COMPLETE.json', folder / 'settings.json', folder / 'results.csv']
        if not all(p.exists() for p in needed):
            continue
        s = read_json(folder / 'settings.json')
        method = str(s.get('method'))
        if method not in METHODS:
            continue
        order_id, seed = int(s['order_id']), int(s['seed'])
        if order_id not in ORDERS or seed not in SEEDS:
            continue
        found.append({'run_name': s.get('run_name', folder.name), 'method': method, 'order_id': order_id,
                      'seed': seed, 'order': list(s['order']), 'folder': folder,
                      'stage1_from': s.get('stage1_from'), 'source': source,
                      'protocol_version': str(s.get('protocol_version')), 'data_version': str(s.get('data_version')),
                      'model_revision': str(s.get('model_revision'))})
    return found

primary = scan_runs(GROUP26_ARTIFACTS, 'group26')
extended = scan_runs(EXTENDED_ARTIFACTS, 'extended') if EXTENDED_ARTIFACTS.exists() else []
records = {}
for rec in primary:
    records[(rec['method'], rec['order_id'], rec['seed'])] = rec
for rec in extended:
    key = (rec['method'], rec['order_id'], rec['seed'])
    if rec['method'] in EXTENDED_FILL_METHODS and key not in records:
        records[key] = rec
if not records:
    raise FileNotFoundError(f'No completed runs found under {GROUP26_ARTIFACTS}/runs')

inventory = (pd.DataFrame(list(records.values()))
             .groupby('method').agg(cells=('run_name', 'count'), source=('source', lambda s: ', '.join(sorted(set(s)))),
                                    protocol=('protocol_version', lambda s: ', '.join(sorted(set(s)))),
                                    data=('data_version', lambda s: ', '.join(sorted(set(s)))))
             .reindex(METHODS).dropna(how='all').reset_index())
inventory['complete'] = inventory['cells'] == len(ORDERS) * len(SEEDS)
show(inventory.assign(method=inventory['method'].map(MLABEL)), 'T00_inventory', 'Runs found per method')

USED = [m for m in METHODS if m in set(inventory.loc[inventory['complete'] | ALLOW_PARTIAL, 'method'])]
missing = [m for m in METHODS if m not in USED]
print('Methods used:', ', '.join(MLABEL[m] for m in USED))
if missing:
    print('Not used (fewer than 8 cells or not found):', ', '.join(MLABEL[m] for m in missing))
for need in ['none', BASELINE]:
    assert need in USED, f'{MLABEL[need]} must have all 8 cells'
records = {k: v for k, v in records.items() if k[0] in USED}
for rec in records.values():
    if PROTOCOL_ORDERS:
        assert rec['order'] == PROTOCOL_ORDERS[rec['order_id']], f"order mismatch in {rec['run_name']}"

### 1.1 Are the comparisons fair?

Three checks before any number is trusted:

1. Every run uses the same data version and model revision.
2. Every method in a cell starts from the **identical Stage-1 model**: its Stage-1 validation margins
   must match the No replay run of that cell. This matters most for runs read from the extended folder.
3. The accuracy recomputed from the pair-level margins equals `results.csv` (checked in section 5).

In [ ]:
def stage_dir(rec, stage):
    d = rec['order'][stage - 1]
    direct = rec['folder'] / f'stage{stage}_{d}'
    if direct.exists() or stage != 1 or not rec['stage1_from']:
        return direct
    source = Path(rec['stage1_from'])
    if not source.exists():
        source = rec['folder'].parent / source.name
    return source / f'stage1_{d}'

def margins(rec, stage, dataset):
    path = stage_dir(rec, stage) / f'margins_{dataset}_{SPLIT}.csv'
    return pd.read_csv(path)

meta = pd.DataFrame(list(records.values()))
for column in ['data_version', 'model_revision', 'protocol_version']:
    values = sorted(meta[column].unique())
    flag = 'OK' if len(values) == 1 else 'CHECK: mixed values'
    print(f'{column:17s}: {values}  -> {flag}')

stage1_rows = []
for (method, order_id, seed), rec in records.items():
    if method == 'none':
        continue
    base = records[('none', order_id, seed)]
    first = rec['order'][0]
    a = margins(rec, 1, first)[['id', MARGIN_COL]].rename(columns={MARGIN_COL: 'margin'})
    b = margins(base, 1, first)[['id', MARGIN_COL]].rename(columns={MARGIN_COL: 'margin'})
    q = a.merge(b, on='id', suffixes=('', '_none'))
    stage1_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'source': rec['source'],
                        'pairs_matched': len(q), 'max_abs_margin_diff': float((q['margin'] - q['margin_none']).abs().max())})
stage1_check = pd.DataFrame(stage1_rows)
bad = stage1_check[(stage1_check['max_abs_margin_diff'] > 1e-4) | (stage1_check['pairs_matched'] < 200)]
summary_check = (stage1_check.groupby('method').agg(cells=('seed', 'count'), worst_diff=('max_abs_margin_diff', 'max'))
                 .reindex(USED).dropna().reset_index())
show(summary_check.assign(method=summary_check['method'].map(MLABEL)), 'T00_stage1_check',
     'Stage-1 identity check (worst difference from the No replay Stage-1 margins)', decimals=6)
if len(bad):
    warnings.warn(f'{len(bad)} runs do NOT share the No replay Stage-1 model. Inspect `bad` before reporting.')
    display(bad)
else:
    print('All methods share the identical Stage-1 model in every cell.')

## 2. Per-run summary

For each run (one method in one order–seed cell):

- **Old-behavior change (retention):** for each of the first two behaviors, final accuracy minus
  accuracy right after it was learned; averaged over the two. Closer to 0 is better.
- **Final task:** accuracy on the behavior learned last (new learning).
- **Final average:** mean final accuracy over Helpful, Safe, and Quality (the overall score).

In [ ]:
results = {key: pd.read_csv(rec['folder'] / 'results.csv') for key, rec in records.items()}

def acc(frame, stage, dataset, score=SCORE):
    row = frame[(frame['stage'] == stage) & (frame['eval_set'] == dataset)]
    return float(row.iloc[0][score]) if len(row) else np.nan

run_rows, behavior_rows = [], []
for (method, order_id, seed), rec in records.items():
    frame, order, last = results[(method, order_id, seed)], rec['order'], len(rec['order'])
    changes = []
    for learned_at, d in enumerate(order[:-1], 1):
        learned, final = acc(frame, learned_at, d), acc(frame, last, d)
        changes.append(final - learned)
        behavior_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'dataset': d,
                              'learned_at': learned_at, 'learned': learned, 'final': final, 'change': final - learned})
    finals = {d: acc(frame, last, d) for d in DATASETS}
    gains = [acc(frame, s, d) - acc(frame, s - 1, d) for s, d in enumerate(order, 1)]
    run_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'old_change': np.mean(changes),
                     'worst_change': np.min(changes), 'final_task': finals[order[-1]],
                     'final_average': np.mean(list(finals.values())), 'mean_stage_gain': np.mean(gains),
                     **{f'final_{d}': finals[d] for d in DATASETS}})
per_run = pd.DataFrame(run_rows)
per_behavior = pd.DataFrame(behavior_rows)

def wide(metric, frame=None):
    frame = per_run if frame is None else frame
    return frame.pivot_table(index=['order_id', 'seed'], columns='method', values=metric).reindex(columns=USED)

## 3. Joint training (upper reference)

Joint training learns all three behaviors together in one stage, so nothing is forgotten by design.
It gives the best final average we could hope for and lets us report the **share of the gap closed**:
`(method − No replay) / (Joint − No replay)` on the final average.

In [ ]:
joint_rows = []
for root in [GROUP26_ARTIFACTS, EXTENDED_ARTIFACTS]:
    candidates = sorted((root / 'joint_runs').glob('*')) if (root / 'joint_runs').exists() else []
    if (root / 'runs').exists():
        candidates += [p for p in sorted((root / 'runs').glob('*joint*'))]
    for folder in candidates:
        if not ((folder / 'results.csv').exists() and (folder / 'COMPLETE.json').exists()):
            continue
        s = read_json(folder / 'settings.json') if (folder / 'settings.json').exists() else {}
        frame = pd.read_csv(folder / 'results.csv')
        last = int(frame['stage'].max())
        seed = int(s.get('seed', frame['seed'].iloc[0] if 'seed' in frame else -1))
        finals = {d: acc(frame, last, d) for d in DATASETS}
        joint_rows.append({'run': folder.name, 'source': 'group26' if root == GROUP26_ARTIFACTS else 'extended',
                           'seed': seed, **finals, 'final_average': np.mean(list(finals.values()))})
joint = pd.DataFrame(joint_rows).drop_duplicates('seed') if joint_rows else pd.DataFrame()
if len(joint):
    matched = joint[joint['seed'].isin(SEEDS)]
    if len(matched):
        joint = matched
    else:
        warnings.warn('No joint run with seeds 0-1; using all joint seeds found.')
    show(joint, 'T00_joint_runs', 'Joint-training runs used')
    JOINT_AVG = float(joint['final_average'].mean())
    JOINT_BY_D = {d: float(joint[d].mean()) for d in DATASETS}
else:
    JOINT_AVG, JOINT_BY_D = np.nan, {}
    print('No joint runs found; gap-closed columns will be empty.')

## 4. Main results (Report §4.3, Table 1)

Means over the 8 cells, sorted **worst to best** by old-behavior change. Brackets are 95% paired
bootstrap intervals over cells for the difference from Random 10% (20,000 resamples). "Cells better"
counts the cells (out of 8) where the method forgot less than Random 10% in the same cell.

In [ ]:
def boot_mean(values, seed=BOOT_SEED):
    v = np.asarray(values, float)
    v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    samples = v[rng.integers(0, len(v), size=(N_BOOT, len(v)))].mean(axis=1)
    lo, hi = np.quantile(samples, [0.025, 0.975])
    return v.mean(), lo, hi

def paired(metric, a, b, seed=BOOT_SEED, frame=None):
    w = wide(metric, frame)
    d = (w[a] - w[b]).dropna().to_numpy()
    mean, lo, hi = boot_mean(d, seed)
    return {'diff': mean, 'lo': lo, 'hi': hi, 'better': int((d > 1e-9).sum()), 'tied': int((np.abs(d) <= 1e-9).sum()),
            'worse': int((d < -1e-9).sum()), 'n': len(d)}

means = per_run.groupby('method')[['old_change', 'worst_change', 'final_task', 'final_average']].mean().reindex(USED)
base_change = means.loc[BASELINE, 'old_change']
none_avg = means.loc['none', 'final_average']
rows = []
for i, m in enumerate(USED):
    r = paired('old_change', m, BASELINE, BOOT_SEED + i) if m != BASELINE else None
    rows.append({'key': m, 'Method': MLABEL[m] + (' †' if m in EXTENSION_METHODS else ''),
                 'Old-behavior change': means.loc[m, 'old_change'],
                 'Less forgetting than Random 10% (%)': 100 * (means.loc[m, 'old_change'] - base_change) / abs(base_change),
                 'Δ retention vs Random 10%': r['diff'] if r else np.nan,
                 '95% low': r['lo'] if r else np.nan, '95% high': r['hi'] if r else np.nan,
                 'Cells better (of 8)': f"{r['better']}/{r['n']}" if r else '—',
                 'Final task': means.loc[m, 'final_task'],
                 'Δ final task vs Random 10%': means.loc[m, 'final_task'] - means.loc[BASELINE, 'final_task'],
                 'Final average': means.loc[m, 'final_average'],
                 'Gap closed to Joint (%)': 100 * (means.loc[m, 'final_average'] - none_avg) / (JOINT_AVG - none_avg)
                 if np.isfinite(JOINT_AVG) and JOINT_AVG != none_avg else np.nan})
table1 = pd.DataFrame(rows).sort_values('Old-behavior change').reset_index(drop=True)
if np.isfinite(JOINT_AVG):
    table1 = pd.concat([table1, pd.DataFrame([{'key': 'joint', 'Method': 'Joint training (reference)',
                                               'Final average': JOINT_AVG, 'Gap closed to Joint (%)': 100.0}])], ignore_index=True)
show(table1.drop(columns='key'), 'T1_main_results',
     'Table 1. Main results (validation, 8 cells). † = changes the replay loss, not only the selection.')

### Figure 1: forgetting per method, worst to best

Bars show mean old-behavior change; whiskers are 95% bootstrap intervals over the 8 cells. The dashed
line marks Random 10%, the equal-budget baseline.

In [ ]:
order_methods = list(table1.loc[table1['key'] != 'joint', 'key'])
fig, ax = plt.subplots(figsize=(FULL_W, 0.36 * len(order_methods) + 0.9))
for y, m in enumerate(order_methods):
    v = wide('old_change')[m].to_numpy()
    mean, lo, hi = boot_mean(v, BOOT_SEED + y)
    ax.barh(y, mean, height=0.62, color=MCOLOR[m])
    ax.errorbar(mean, y, xerr=[[mean - lo], [hi - mean]], color=INK2, lw=1, capsize=2.5)
    pct = 100 * (mean - base_change) / abs(base_change)
    note = 'equal-budget baseline' if m == BASELINE else (f'{pct:.1f}% less forgetting' if pct >= 0 else f'{-pct:.1f}% more forgetting')
    ax.text(1.02, y, f'{mean:.2f}   {note}', transform=ax.get_yaxis_transform(), va='center', ha='left', fontsize=8, color=INK)
ax.axvline(base_change, color=INK2, lw=1, ls='--')
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(range(len(order_methods)), [MLABEL[m] for m in order_methods])
ax.invert_yaxis()
ax.set_xlim(min(wide('old_change').min().min(), -1) * 1.08, 0)
ax.set_xlabel('Old-behavior change after all stages (accuracy points; closer to 0 is better)')
ax.set_title('Forgetting of earlier behaviors, worst to best (mean of 8 cells; vs Random 10% on the right)', loc='left')
clean_axis(ax, 'x')
save(fig, 'F1_forgetting_by_method')

### Figure 2: retention vs new learning (the trade-off)

A method that forgets less but blocks new learning is not useful. Each point is a method's mean over
8 cells with 95% intervals on both axes. The shaded band is more than 2 points below Random 10% on the
final task, which our success rule does not allow.

In [ ]:
fig, ax = plt.subplots(figsize=(FULL_W * 0.85, 3.8))
xr = means.loc[BASELINE, 'final_task']
pts = []
for i, m in enumerate(USED):
    x, xlo, xhi = boot_mean(wide('final_task')[m], BOOT_SEED + 50 + i)
    y, ylo, yhi = boot_mean(wide('old_change')[m], BOOT_SEED + 80 + i)
    ax.errorbar(x, y, xerr=[[x - xlo], [xhi - x]], yerr=[[y - ylo], [yhi - y]], fmt='none', ecolor=MCOLOR[m], lw=0.9, alpha=0.55)
    ax.scatter(x, y, s=55, color=MCOLOR[m], edgecolor='white', linewidth=1.5, zorder=3)
    pts.append((m, x, y))
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.set_xlim(min(x0, xr - TOLERANCE - 0.5), x1 + 0.12 * (x1 - x0))
x0, x1 = ax.get_xlim()
dy, dx = 0.06 * (y1 - y0), 0.30 * (x1 - x0)
placed = []
for m, x, y in sorted(pts, key=lambda p: -p[2]):
    ly = y
    for (px, py) in placed:
        if abs(px - x) < dx and abs(py - ly) < dy:
            ly = py - dy
    placed.append((x, ly))
    ax.annotate(MLABEL[m], (x, y), xytext=(x + 0.015 * (x1 - x0), ly), textcoords='data', fontsize=8, color=INK,
                va='center', zorder=4, bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.9),
                arrowprops=dict(arrowstyle='-', color=INK2, lw=0.6) if abs(ly - y) > 1e-9 else None)
ax.axvspan(x0, xr - TOLERANCE, color=NEUTRAL, zorder=0)
ax.axvline(xr - TOLERANCE, color=INK2, lw=0.8, ls=':')
ax.text(xr - TOLERANCE - 0.05, y1 - 0.03 * (y1 - y0), 'over 2 pts below\nRandom 10%\non new task', ha='right', va='top', fontsize=7, color=INK2)
ax.set_xlabel('Final-task accuracy (%) — new learning')
ax.set_ylabel('Old-behavior change (points; up = less forgetting)')
ax.set_title('Retention vs new learning (mean of 8 cells, 95% intervals)', loc='left')
clean_axis(ax, 'both')
save(fig, 'F2_retention_vs_new_learning')

## 5. Statistics: every method vs Random 10%, MFR, and Lowest margin (Report §4.3, Table 2)

Paired by cell: each difference is method minus comparison in the **same** order and seed.
Positive = the method is better for all three metrics. "Excludes 0" means the 95% interval does not
contain zero.

In [ ]:
stat_rows = []
for c_i, comp in enumerate([BASELINE, 'mfr', 'lowest_margin']):
    if comp not in USED:
        continue
    for m_i, m in enumerate(USED):
        if m == comp:
            continue
        for k_i, (metric, label) in enumerate([('old_change', 'Retention'), ('final_task', 'Final task'),
                                               ('final_average', 'Final average')]):
            r = paired(metric, m, comp, BOOT_SEED + 1000 * c_i + 10 * m_i + k_i)
            stat_rows.append({'Method': MLABEL[m], 'vs': MLABEL[comp], 'Metric': label, 'Difference': r['diff'],
                              '95% low': r['lo'], '95% high': r['hi'],
                              'Excludes 0': 'yes' if r['lo'] > 0 or r['hi'] < 0 else 'no',
                              'Better / tied / worse cells': f"{r['better']} / {r['tied']} / {r['worse']}"})
stats = pd.DataFrame(stat_rows)
show(stats[stats['vs'] == MLABEL[BASELINE]], 'T2a_paired_vs_random', 'Table 2a. Paired differences vs Random 10%')
show(stats[stats['vs'] != MLABEL[BASELINE]], 'T2b_paired_vs_mfr_and_lowest_margin',
     'Table 2b. Paired differences vs MFR and vs Lowest margin')

### Figure 3: paired differences from Random 10%

Points right of zero are better than equal-budget random replay. Left panel: retention. Right panel:
final three-behavior average.

In [ ]:
vs = stats[stats['vs'] == MLABEL[BASELINE]]
methods_vs = [m for m in order_methods if m != BASELINE]
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 0.34 * len(methods_vs) + 1.0), sharey=True)
for ax, metric in zip(axes, ['Retention', 'Final average']):
    part = vs[vs['Metric'] == metric].set_index('Method')
    for y, m in enumerate(methods_vs):
        r = part.loc[MLABEL[m]]
        ax.errorbar(r['Difference'], y, xerr=[[r['Difference'] - r['95% low']], [r['95% high'] - r['Difference']]],
                    fmt='o', color=MCOLOR[m], ms=6, mec='white', mew=1.2, lw=1.2, capsize=2.5)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_title(f'{metric}: method − Random 10%', loc='left')
    ax.set_xlabel('Accuracy points (right = better)')
    clean_axis(ax, 'x')
axes[0].set_yticks(range(len(methods_vs)), [MLABEL[m] for m in methods_vs])
axes[0].invert_yaxis()
save(fig, 'F3_paired_vs_random')

## 6. Forgetting per behavior (Report §4.3, Table 3)

Helpful and Safe are earlier behaviors in 6 of 8 cells; Quality in 4 (it is learned first only in
orders 3–4 and is never in the middle; state this as a limitation).

In [ ]:
pb = per_behavior.groupby(['method', 'dataset'])['change'].agg(['mean', 'count']).reset_index()
t3 = pb.pivot(index='method', columns='dataset', values='mean').reindex(order_methods).reindex(columns=DATASETS)
counts = pb.pivot(index='method', columns='dataset', values='count').max()
t3_out = t3.reset_index().rename(columns={'method': 'Method', **{d: f'{DLABEL[d]} (n={int(counts[d])})' for d in DATASETS}})
t3_out['Method'] = t3_out['Method'].map(MLABEL)
t3_out['Worst behavior'] = t3.min(axis=1).to_numpy()
show(t3_out, 'T3_forgetting_by_behavior', 'Table 3. Old-behavior change by behavior (mean over cells where it is an earlier behavior)')

fig, ax = plt.subplots(figsize=(FULL_W, 3.0))
x = np.arange(len(order_methods)); w = 0.26
for i, d in enumerate(DATASETS):
    ax.bar(x + (i - 1) * w, t3[d].to_numpy(), w * 0.92, color=DCOLOR[d], label=DLABEL[d])
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(x, [MLABEL[m] for m in order_methods], rotation=20, ha='right')
ax.set_ylabel('Accuracy change (points)')
ax.set_title('Forgetting by behavior (closer to 0 is better)', loc='left', pad=24)
ax.legend(ncol=3, loc='lower center', bbox_to_anchor=(0.5, 1.06))
clean_axis(ax, 'y')
save(fig, 'F4_forgetting_by_behavior')

### Final accuracy per behavior and the gap to joint training (Report §4.3)

In [ ]:
fin = per_run.groupby('method')[[f'final_{d}' for d in DATASETS] + ['final_average']].mean().reindex(order_methods)
fin_out = fin.reset_index().rename(columns={'method': 'Method', **{f'final_{d}': DLABEL[d] for d in DATASETS},
                                            'final_average': 'Average'})
fin_out['Method'] = fin_out['Method'].map(MLABEL)
if np.isfinite(JOINT_AVG):
    fin_out = pd.concat([fin_out, pd.DataFrame([{'Method': 'Joint training (reference)',
                                                 **{DLABEL[d]: JOINT_BY_D[d] for d in DATASETS}, 'Average': JOINT_AVG}])],
                        ignore_index=True)
show(fin_out, 'T4_final_accuracy_by_behavior', 'Table 4. Final accuracy per behavior after all three stages')

## 7. Does a method protect what was learned, or fix what was weak? (Report §3.6, §4.3)

For each earlier behavior we compare every validation pair **right after it was learned** with the
**end of training**. A pair is correct when its margin is above 0 (the model prefers the chosen answer
more than the base model does).

| | Correct at end | Incorrect at end |
|---|---|---|
| **Correct when learned** | Kept | **Forgotten** |
| **Incorrect when learned** | **Recovered** | Never learned |

- **Forgotten share** and **recovered share** are percentages of all 200 pairs, so
  `net change = recovered share − forgotten share` (this equals the old-behavior change).
- **Forgetting rate** = forgotten / pairs correct when learned. **Recovery rate** = recovered / pairs
  incorrect when learned. Their denominators differ, so they do not add up.

Validation pairs are never replayed (the buffer comes from the training split), so this measures
whether protection **generalizes**.

In [ ]:
trans_rows, pair_frames = [], []
for (method, order_id, seed), rec in records.items():
    order, last = rec['order'], len(rec['order'])
    for learned_at, d in enumerate(order[:-1], 1):
        a = margins(rec, learned_at, d)[['id', MARGIN_COL]].rename(columns={MARGIN_COL: 'learned'})
        z = margins(rec, last, d)[['id', MARGIN_COL]].rename(columns={MARGIN_COL: 'final'})
        q = a.merge(z, on='id')
        cl, cf = q['learned'] > 0, q['final'] > 0
        n = len(q)
        kept, forgotten = int((cl & cf).sum()), int((cl & ~cf).sum())
        recovered, never = int((~cl & cf).sum()), int((~cl & ~cf).sum())
        trans_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'dataset': d, 'n': n,
                           'kept': kept, 'forgotten': forgotten, 'recovered': recovered, 'never': never,
                           'forgotten_share': 100 * forgotten / n, 'recovered_share': 100 * recovered / n,
                           'net_change': 100 * (recovered - forgotten) / n,
                           'forgetting_rate': 100 * forgotten / max(kept + forgotten, 1),
                           'recovery_rate': 100 * recovered / max(recovered + never, 1),
                           'margin_change_known': float((q['final'] - q['learned'])[cl].mean())})
        q = q.assign(method=method, order_id=order_id, seed=seed, dataset=d)
        pair_frames.append(q)
transitions = pd.DataFrame(trans_rows)
pairs = pd.concat(pair_frames, ignore_index=True)

check = transitions.merge(per_behavior, on=['method', 'order_id', 'seed', 'dataset'])
mismatch = check[(check['net_change'] - check['change']).abs() > 0.051]
print('Margin-level accuracy matches results.csv for every run and behavior.' if mismatch.empty
      else f'WARNING: {len(mismatch)} run-behavior rows disagree with results.csv')

per_run_t = transitions.groupby(['method', 'order_id', 'seed'])[
    ['forgotten_share', 'recovered_share', 'net_change', 'forgetting_rate', 'recovery_rate', 'margin_change_known']].mean().reset_index()
t5 = per_run_t.groupby('method')[['forgotten_share', 'recovered_share', 'net_change', 'forgetting_rate',
                                  'recovery_rate', 'margin_change_known']].mean().reindex(order_methods)
rows = []
for i, m in enumerate(order_methods):
    f = paired('forgotten_share', m, BASELINE, BOOT_SEED + 300 + i, per_run_t) if m != BASELINE else None
    r = paired('recovered_share', m, BASELINE, BOOT_SEED + 400 + i, per_run_t) if m != BASELINE else None
    rows.append({'Method': MLABEL[m], 'Forgotten (% of pairs)': t5.loc[m, 'forgotten_share'],
                 'Recovered (% of pairs)': t5.loc[m, 'recovered_share'], 'Net change': t5.loc[m, 'net_change'],
                 'Forgetting rate (% of known)': t5.loc[m, 'forgetting_rate'],
                 'Recovery rate (% of unknown)': t5.loc[m, 'recovery_rate'],
                 'Δ forgotten vs Random [95%]': f"{f['diff']:+.2f} [{f['lo']:+.2f}, {f['hi']:+.2f}]" if f else '—',
                 'Δ recovered vs Random [95%]': f"{r['diff']:+.2f} [{r['lo']:+.2f}, {r['hi']:+.2f}]" if r else '—',
                 'Mean margin change of known pairs': t5.loc[m, 'margin_change_known']})
table5 = pd.DataFrame(rows)
show(table5, 'T5_transitions', 'Table 5. Where the net change comes from (mean over 8 cells × 2 earlier behaviors)', decimals=3)

### Figure 5: forgotten vs recovered pairs

Left of zero: pairs that were correct when learned and wrong at the end. Right of zero: pairs that
were wrong when learned and correct at the end. The black dot is the net change.

In [ ]:
fig, ax = plt.subplots(figsize=(FULL_W, 0.36 * len(order_methods) + 1.0))
for y, m in enumerate(order_methods):
    f, r, n = t5.loc[m, 'forgotten_share'], t5.loc[m, 'recovered_share'], t5.loc[m, 'net_change']
    ax.barh(y, -f, height=0.6, color=FORGET_C)
    ax.barh(y, r, height=0.6, color=RECOVER_C)
    ax.plot(n, y, 'D', color=INK, ms=4.5, zorder=3, mec='white', mew=0.8)
    ax.annotate(f'{f:.1f}', (-f, y), xytext=(-5, 0), textcoords='offset points', va='center', ha='right', fontsize=8)
    ax.annotate(f'{r:.1f}', (r, y), xytext=(5, 0), textcoords='offset points', va='center', ha='left', fontsize=8)
ax.axvline(0, color=INK2, lw=1)
ax.set_yticks(range(len(order_methods)), [MLABEL[m] for m in order_methods])
ax.invert_yaxis()
ax.set_xlim(-(t5['forgotten_share'].max() * 1.15 + 1), max(t5['recovered_share'].max() * 1.25, 3) + 1)
ax.set_xlabel('% of validation pairs   (left: forgotten   |   right: recovered;   diamond: net change)')
ax.set_title('Forgotten vs recovered pairs of earlier behaviors (mean of 8 cells)', loc='left')
clean_axis(ax, 'x')
save(fig, 'F5_forgotten_vs_recovered')

### Figure 6: the full 2×2 transition table per method (appendix)

Percent of all validation pairs of earlier behaviors, pooled over cells.

In [ ]:
pool = transitions.groupby('method')[['kept', 'forgotten', 'recovered', 'never', 'n']].sum().reindex(order_methods)
ncol = min(5, len(order_methods)); nrow = math.ceil(len(order_methods) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(FULL_W, 1.6 * nrow + 0.4), gridspec_kw={'wspace': 0.12, 'hspace': 0.45})
axes = np.atleast_1d(axes).ravel()
for idx, (ax, m) in enumerate(zip(axes, order_methods)):
    p = pool.loc[m]
    mat = 100 * np.array([[p['kept'], p['forgotten']], [p['recovered'], p['never']]]) / p['n']
    ax.imshow(mat, cmap=SEQUENTIAL, vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{mat[i, j]:.1f}', ha='center', va='center', fontsize=8,
                    color='white' if mat[i, j] > 45 else INK)
    ax.set_xticks([0, 1], ['correct', 'wrong'], fontsize=7)
    ax.set_yticks([0, 1], ['correct', 'wrong'] if idx % ncol == 0 else ['', ''], fontsize=7)
    ax.set_title(MLABEL[m], fontsize=8.5)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
for ax in axes[len(order_methods):]:
    ax.axis('off')
fig.supxlabel('At the end of training', fontsize=8.5)
fig.supylabel('When learned', fontsize=8.5)
save(fig, 'F6_transition_matrices')

## 8. Are weakly learned pairs forgotten differently? (Report §4.3, discussion)

MFR ranks pairs by how far their margin **dropped**. A pair that was learned only weakly cannot drop
far, so MFR rarely picks it, even if it is about to flip. Lowest margin picks exactly those pairs.
To test this, the pairs that were correct when learned are split into quartiles of their learned
margin within each run and behavior (Q1 = weakest correct pairs, Q4 = strongest). For each quartile we
report the **forgetting rate** and the **mean margin drop**.

In [ ]:
known = pairs[pairs['learned'] > 0].copy()
known['quartile'] = (known.groupby(['method', 'order_id', 'seed', 'dataset'])['learned']
                     .transform(lambda s: pd.qcut(s.rank(method='first'), 4, labels=False) + 1))
known['forgot'] = (known['final'] <= 0).astype(float)
known['drop'] = known['learned'] - known['final']
quart = (known.groupby(['method', 'order_id', 'seed', 'dataset', 'quartile'])
         .agg(rate=('forgot', 'mean'), drop=('drop', 'mean'), learned=('learned', 'mean')).reset_index()
         .groupby(['method', 'quartile'])[['rate', 'drop', 'learned']].mean().reset_index())
quart['rate'] *= 100
q_rate = quart.pivot(index='method', columns='quartile', values='rate').reindex(order_methods)
q_drop = quart.pivot(index='method', columns='quartile', values='drop').reindex(order_methods)
t6 = pd.DataFrame({'Method': [MLABEL[m] for m in order_methods]})
for qq in [1, 2, 3, 4]:
    t6[f'Q{qq} forgetting rate (%)'] = q_rate[qq].to_numpy()
for qq in [1, 2, 3, 4]:
    t6[f'Q{qq} margin drop'] = q_drop[qq].to_numpy()
show(t6, 'T6_forgetting_by_learned_margin', 'Table 6. Forgetting by learned-margin quartile (Q1 = weakest correct pairs)', decimals=3)

key = [m for m in KEY_METHODS if m in USED]
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.9))
for m in key:
    axes[0].plot([1, 2, 3, 4], q_rate.loc[m, [1, 2, 3, 4]], marker='o', ms=5, color=MCOLOR[m], label=MLABEL[m])
    axes[1].plot([1, 2, 3, 4], q_drop.loc[m, [1, 2, 3, 4]], marker='o', ms=5, color=MCOLOR[m], label=MLABEL[m])
axes[0].set_ylabel('Forgetting rate (% of pairs)')
axes[1].set_ylabel('Mean margin drop')
for ax, title in zip(axes, ['Forgetting rate by learned margin', 'Margin drop by learned margin']):
    ax.set_xticks([1, 2, 3, 4], ['Q1\nweakest', 'Q2', 'Q3', 'Q4\nstrongest'])
    ax.set_title(title, loc='left')
    clean_axis(ax, 'y')
fig.supxlabel('Quartile of the margin right after learning (only pairs that were correct then)', fontsize=8.5, y=-0.07)
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=len(key), bbox_to_anchor=(0.5, 1.0), fontsize=8)
save(fig, 'F7_forgetting_by_learned_margin')

## 9. How an early behavior fades across stages (Report §4.2 / §4.3)

For each order, the accuracy of the **first** behavior after stages 1, 2, and 3 (mean over 2 seeds).
Stage 1 is identical for every method.

In [ ]:
curve_rows = []
for (method, order_id, seed), rec in records.items():
    first = rec['order'][0]
    for stage in [1, 2, 3]:
        curve_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'stage': stage,
                           'accuracy': acc(results[(method, order_id, seed)], stage, first)})
curves = pd.DataFrame(curve_rows).groupby(['order_id', 'method', 'stage'])['accuracy'].mean().reset_index()
fig, axes = plt.subplots(1, len(ORDERS), figsize=(FULL_W, 2.6), sharey=True, gridspec_kw={'wspace': 0.3})
for ax, o in zip(np.atleast_1d(axes), ORDERS):
    order = PROTOCOL_ORDERS.get(o) or next(r['order'] for r in records.values() if r['order_id'] == o)
    for m in key:
        p = curves[(curves['order_id'] == o) & (curves['method'] == m)].sort_values('stage')
        ax.plot(p['stage'], p['accuracy'], marker='o', ms=4, color=MCOLOR[m], label=MLABEL[m])
    ax.set_xticks([1, 2, 3], [DLABEL[d] for d in order], fontsize=7)
    ax.set_xlim(0.75, 3.25)
    ax.set_title(f'Order {o}: {DLABEL[order[0]]} first', fontsize=8.5, loc='left')
    clean_axis(ax, 'y')
np.atleast_1d(axes)[0].set_ylabel('Accuracy of first behavior (%)')
fig.supxlabel('Behavior just trained (stage 1 → 3)', fontsize=8.5, y=-0.06)
fig.legend(*np.atleast_1d(axes)[0].get_legend_handles_labels(), loc='lower center', ncol=len(key), bbox_to_anchor=(0.5, 1.0), fontsize=8)
save(fig, 'F8_first_behavior_across_stages')

ex_o, ex_s = EXAMPLE_CELL
ex = results.get(('none', ex_o, ex_s))
if ex is not None:
    ex_order = records[('none', ex_o, ex_s)]['order']
    if 'safe' in ex_order[:-1]:
        k = ex_order.index('safe') + 1
        print(f"Example (order {ex_o}, seed {ex_s}, No replay): Safety {acc(ex, k, 'safe'):.1f}% right after learning "
              f"→ {acc(ex, len(ex_order), 'safe'):.1f}% at the end "
              f"({acc(ex, len(ex_order), 'safe') - acc(ex, k, 'safe'):+.1f} points).")
        for stage in range(k, len(ex_order) + 1):
            print(f'   after stage {stage} ({DLABEL[ex_order[stage - 1]]}): Safety = {acc(ex, stage, "safe"):.1f}%')

## 10. What did each method replay? (mechanism; Report §4.3 discussion and §5.1)

From the replay logs of stages 2–3. All methods store the **same** buffer pairs in a cell, so
overlap between methods is meaningful.

- **Unique pairs** and **uses per pair**: how spread out the replay was.
- **Top-10% share**: share of replay slots taken by the 10% most-replayed pairs (concentration).
- **Slots per source behavior**: whether a method spends its budget mostly on one behavior.

In [ ]:
log_rows, id_sets = [], {}
for (method, order_id, seed), rec in records.items():
    if method == 'none':
        continue
    for stage in range(2, len(rec['order']) + 1):
        path = stage_dir(rec, stage) / 'replay_log.csv'
        if not path.exists():
            continue
        log = pd.read_csv(path)
        if log.empty:
            continue
        counts_id = log['id'].value_counts()
        top = int(math.ceil(0.10 * len(counts_id)))
        src = log['dataset'].value_counts(normalize=True)
        log_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'stage': stage, 'slots': len(log),
                         'unique': len(counts_id), 'uses_per_pair': len(log) / len(counts_id),
                         'top10_share': 100 * counts_id.iloc[:top].sum() / len(log),
                         **{f'share_{d}': 100 * float(src.get(d, 0.0)) for d in DATASETS}})
        id_sets[(method, order_id, seed, stage)] = set(log['id'])
replay = pd.DataFrame(log_rows)
if len(replay):
    rep_methods = [m for m in order_methods if m in set(replay['method'])]
    t7 = replay.groupby('method')[['slots', 'unique', 'uses_per_pair', 'top10_share']].mean().reindex(rep_methods)
    t7_out = t7.reset_index().rename(columns={'method': 'Method', 'slots': 'Replay slots per stage',
                                              'unique': 'Unique pairs per stage', 'uses_per_pair': 'Uses per pair',
                                              'top10_share': 'Top-10% pairs share of slots (%)'})
    t7_out['Method'] = t7_out['Method'].map(MLABEL)
    show(t7_out, 'T7_replay_concentration', 'Table 7. How concentrated replay was (stages 2–3, mean over runs)')

    # Allocation across source behaviors in stage 3 (two old behaviors available), per order.
    alloc = replay[replay['stage'] == 3].copy()
    alloc_rows = []
    for (m, o), g in alloc.groupby(['method', 'order_id']):
        old = (PROTOCOL_ORDERS.get(o) or next(r['order'] for r in records.values() if r['order_id'] == o))[:2]
        alloc_rows.append({'method': m, 'order_id': o, 'first': old[0], 'second': old[1],
                           'share_first': g[f'share_{old[0]}'].mean(), 'share_second': g[f'share_{old[1]}'].mean()})
    alloc = pd.DataFrame(alloc_rows)
    safe_share = []
    for m in rep_methods:
        g = alloc[alloc['method'] == m]
        s = [r['share_first'] if r['first'] == 'safe' else r['share_second'] for _, r in g.iterrows() if 'safe' in (r['first'], r['second'])]
        safe_share.append(np.mean(s) if s else np.nan)
    t7b = pd.DataFrame({'Method': [MLABEL[m] for m in rep_methods], 'Stage-3 slots spent on Safe (%)': safe_share,
                        'Equal split would be (%)': 50.0})
    show(t7b, 'T7b_safe_share', 'Table 7b. Share of stage-3 replay spent on Safe, when Safe is an old behavior')

    # Jaccard overlap of the pairs chosen by each method (same cell and stage).
    jac = pd.DataFrame(np.nan, index=rep_methods, columns=rep_methods)
    for a in rep_methods:
        for b in rep_methods:
            vals = [len(id_sets[(a, o, s, st)] & id_sets[(b, o, s, st)]) / len(id_sets[(a, o, s, st)] | id_sets[(b, o, s, st)])
                    for (mm, o, s, st) in id_sets if mm == a and (b, o, s, st) in id_sets]
            jac.loc[a, b] = np.mean(vals) if vals else np.nan
    jac_out = jac.copy(); jac_out.index = [MLABEL[m] for m in jac.index]; jac_out.columns = [MLABEL[m] for m in jac.columns]
    show(jac_out.reset_index().rename(columns={'index': 'Method'}), 'T8_selection_overlap',
         'Table 8. Overlap (Jaccard) of the replayed pairs, same cell and stage')

    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 0.34 * len(rep_methods) + 1.3),
                             gridspec_kw={'width_ratios': [1, 1.15]})
    ax = axes[0]
    for y, m in enumerate(rep_methods):
        ax.barh(y, t7.loc[m, 'top10_share'], height=0.6, color=MCOLOR[m])
        ax.text(t7.loc[m, 'top10_share'] + 1, y, f"{t7.loc[m, 'top10_share']:.0f}%  ({t7.loc[m, 'uses_per_pair']:.1f} uses/pair)",
                va='center', fontsize=7.5)
    ax.axvline(10, color=INK2, lw=0.8, ls='--')
    ax.set_yticks(range(len(rep_methods)), [MLABEL[m] for m in rep_methods]); ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel('Slots taken by the 10% most-replayed pairs (%)')
    ax.set_title('Replay concentration (dashed = uniform)', loc='left', fontsize=9)
    clean_axis(ax, 'x')
    ax = axes[1]
    im = ax.imshow(jac.to_numpy(dtype=float), cmap=SEQUENTIAL, vmin=0, vmax=1)
    for i in range(len(rep_methods)):
        for j in range(len(rep_methods)):
            v = jac.iloc[i, j]
            ax.text(j, i, '' if np.isnan(v) else f'{v:.2f}', ha='center', va='center', fontsize=6.5,
                    color='white' if (not np.isnan(v) and v > 0.55) else INK)
    ax.set_xticks(range(len(rep_methods)), [MLABEL[m] for m in rep_methods], rotation=40, ha='right', fontsize=7)
    ax.set_yticks(range(len(rep_methods)), [MLABEL[m] for m in rep_methods], fontsize=7)
    ax.set_title('Overlap of replayed pairs (Jaccard)', loc='left', fontsize=9)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    save(fig, 'F9_replay_mechanism')
else:
    print('No replay logs found.')

## 11. Cost (Report §4.1 / §5)

Minutes per run from the training histories. Stage 1 is shared, so the comparison that matters is
stages 2–3. Scoring minutes are the extra time spent re-scoring the buffer (margin-based methods).

In [ ]:
cost_rows = []
for (method, order_id, seed), rec in records.items():
    later = scoring = 0.0; replayed = 0
    for stage in range(2, len(rec['order']) + 1):
        path = stage_dir(rec, stage) / 'history.csv'
        if not path.exists():
            continue
        h = pd.read_csv(path)
        later += float(h['minutes'].max())
        scoring += float(h['scoring_minutes'].max()) if 'scoring_minutes' in h else 0.0
        replayed += int(h['n_replay'].sum()) if 'n_replay' in h else 0
    cost_rows.append({'method': method, 'order_id': order_id, 'seed': seed, 'minutes_stage23': later,
                      'scoring_minutes': scoring, 'replay_examples': replayed})
cost = pd.DataFrame(cost_rows).groupby('method')[['minutes_stage23', 'scoring_minutes', 'replay_examples']].mean().reindex(order_methods)
cost_out = cost.reset_index().rename(columns={'method': 'Method', 'minutes_stage23': 'Minutes, stages 2–3',
                                              'scoring_minutes': 'of which buffer re-scoring',
                                              'replay_examples': 'Replayed examples'})
cost_out['Method'] = cost_out['Method'].map(MLABEL)
cost_out['vs Random 10% (×)'] = (cost['minutes_stage23'] / cost.loc[BASELINE, 'minutes_stage23']).to_numpy()
show(cost_out, 'T9_cost', 'Table 9. Training cost per run (mean over 8 cells; L40S)')

fig, ax = plt.subplots(figsize=(FULL_W * 0.8, 0.32 * len(order_methods) + 0.9))
for y, m in enumerate(order_methods):
    ax.barh(y, cost.loc[m, 'minutes_stage23'], height=0.6, color=MCOLOR[m])
    ax.barh(y, cost.loc[m, 'scoring_minutes'], height=0.6, color=INK, alpha=0.25)
    ax.text(cost.loc[m, 'minutes_stage23'] + 0.5, y, f"{cost.loc[m, 'minutes_stage23']:.0f} min", va='center', fontsize=7.5)
ax.set_yticks(range(len(order_methods)), [MLABEL[m] for m in order_methods]); ax.invert_yaxis()
ax.set_xlabel('Minutes for stages 2–3 (dark part: buffer re-scoring)')
ax.set_title('Training time per run', loc='left')
clean_axis(ax, 'x')
save(fig, 'F10_runtime')

## 12. Every cell (appendix; shows the result is not driven by one order or seed)

In [ ]:
cells = per_run[['order_id', 'seed']].drop_duplicates().sort_values(['order_id', 'seed'])
labels = [f'O{o} S{s}' for o, s in cells.itertuples(index=False)]
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 0.3 * len(labels) + 1.8))
for ax, metric, title, cmap, norm in [
    (axes[0], 'old_change', 'Old-behavior change', DIVERGING, None),
    (axes[1], 'final_average', 'Final average (%)', SEQUENTIAL, None)]:
    mat = wide(metric).reindex(columns=order_methods).to_numpy(dtype=float)
    if metric == 'old_change':
        norm = TwoSlopeNorm(vmin=min(np.nanmin(mat), -0.01), vcenter=0.0, vmax=max(np.nanmax(mat), 0.01))
        im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    else:
        im = ax.imshow(mat, aspect='auto', cmap=cmap)
    shade = im.norm(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            dark = (metric == 'final_average' and shade[i, j] > 0.6)
            ax.text(j, i, f'{mat[i, j]:.1f}', ha='center', va='center', fontsize=6.5, color='white' if dark else INK)
    ax.set_xticks(range(len(order_methods)), [MLABEL[m] for m in order_methods], rotation=40, ha='right', fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7.5)
    ax.set_title(title, loc='left', fontsize=9)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
save(fig, 'F11_every_cell')
for metric, name in [('old_change', 'T10a_every_cell_retention'), ('final_average', 'T10b_every_cell_final_average'),
                     ('final_task', 'T10c_every_cell_final_task')]:
    t = wide(metric).reindex(columns=order_methods).reset_index()
    t = t.rename(columns={'order_id': 'Order', 'seed': 'Seed', **{m: MLABEL[m] for m in order_methods}})
    show(t, name, f'Every cell: {metric}')

## 13. Method-selection rule (Report §5.2; frozen on Oct 30)

The predeclared rule, on seeds 0–1 validation only:

1. mean old-behavior change better than Random 10%;
2. mean final-task accuracy no more than 2 points below Random 10%;
3. among those that pass, the highest final average wins;
4. if the top two are within 0.25 points, the cheaper one wins.

In [ ]:
# Only selection rules are candidates. Extensions (DAPR) are reported next to them but cannot be selected,
# because they change the training loss; set INCLUDE_EXTENSIONS_IN_RULE = True to include them.
INCLUDE_EXTENSIONS_IN_RULE = False
candidates = [m for m in ['mfr', 'mfr_balanced', 'mfr_at_risk', 'mfr_stochastic', 'lowest_margin', 'dapr_weak'] if m in USED]
sel = []
for m in candidates:
    better = means.loc[m, 'old_change'] > means.loc[BASELINE, 'old_change']
    within = means.loc[m, 'final_task'] >= means.loc[BASELINE, 'final_task'] - TOLERANCE
    sel.append({'key': m, 'Method': MLABEL[m] + (' †' if m in EXTENSION_METHODS else ''),
                'Retains better than Random 10%': 'yes' if better else 'no',
                'Final task within 2 points': 'yes' if within else 'no', 'Passes': 'yes' if better and within else 'no',
                'Final average': means.loc[m, 'final_average'], 'Minutes, stages 2–3': cost.loc[m, 'minutes_stage23'],
                'Candidate': 'no (extension)' if (m in EXTENSION_METHODS and not INCLUDE_EXTENSIONS_IN_RULE) else 'yes'})
sel = pd.DataFrame(sel).sort_values('Final average', ascending=False)
passing = sel[(sel['Passes'] == 'yes') & (sel['Candidate'] == 'yes')]
if len(passing):
    top = passing.iloc[0]
    if len(passing) > 1 and top['Final average'] - passing.iloc[1]['Final average'] <= 0.25:
        pair = passing.iloc[:2].sort_values('Minutes, stages 2–3')
        top = pair.iloc[0]
    print(f"Rule selects: {top['Method']} (final average {top['Final average']:.2f})")
show(sel.drop(columns='key'), 'T11_selection_rule', 'Table 11. Method-selection rule applied to seeds 0–1 (validation)')

## 14. Key numbers for the report text

Every number quoted in the report, computed here. Copy sentences from this output; do not retype numbers.

In [ ]:
def pct_less(m, ref=BASELINE):
    return 100 * (means.loc[m, 'old_change'] - means.loc[ref, 'old_change']) / abs(means.loc[ref, 'old_change'])

lines = []
add = lines.append
none_safe = per_behavior[(per_behavior['method'] == 'none') & (per_behavior['dataset'] == 'safe')]['change'].mean()
add(f"Setup: {len(ORDERS)} orders × {len(SEEDS)} seeds = {len(ORDERS) * len(SEEDS)} cells per method; "
    f"{len(records)} runs from {len(USED)} methods; {SPLIT} split, 200 pairs per behavior (1 pair = 0.5 points).")
add(f"Without replay, earlier behaviors lose {abs(means.loc['none', 'old_change']):.2f} points on average; "
    f"Safety loses {abs(none_safe):.2f} points.")
if 'random' in USED:
    add(f"Random 10% replay cuts forgetting by {pct_less('random', 'none'):.1f}% relative to no replay "
        f"({means.loc['none', 'old_change']:.2f} → {means.loc['random', 'old_change']:.2f}).")
for m in [x for x in order_methods if x not in ('none', BASELINE)]:
    r = paired('old_change', m, BASELINE)
    fa = paired('final_average', m, BASELINE)
    pl = pct_less(m)
    add(f"{MLABEL[m]}: {abs(pl):.1f}% {'less' if pl >= 0 else 'more'} forgetting than Random 10% "
        f"(retention {r['diff']:+.2f} points, 95% CI [{r['lo']:.2f}, {r['hi']:.2f}], better in {r['better']}/{r['n']} cells); "
        f"final task {means.loc[m, 'final_task'] - means.loc[BASELINE, 'final_task']:+.2f}; "
        f"final average {means.loc[m, 'final_average']:.2f} ({fa['diff']:+.2f} vs Random 10%, CI [{fa['lo']:.2f}, {fa['hi']:.2f}]).")
if {'mfr', 'lowest_margin'} <= set(USED):
    r = paired('old_change', 'lowest_margin', 'mfr')
    pl = pct_less('lowest_margin', 'mfr')
    add(f"Lowest margin forgets {abs(pl):.1f}% {'less' if pl >= 0 else 'more'} than MFR "
        f"({r['diff']:+.2f} points, CI [{r['lo']:.2f}, {r['hi']:.2f}], better in {r['better']}/{r['n']} cells).")
if {'dapr_weak', 'lowest_margin'} <= set(USED):
    r = paired('old_change', 'dapr_weak', 'lowest_margin')
    fa = paired('final_average', 'dapr_weak', 'lowest_margin')
    add(f"DAPR-weak vs Lowest margin (same selection, plus the anchor): retention {r['diff']:+.2f} "
        f"(CI [{r['lo']:.2f}, {r['hi']:.2f}]), final average {fa['diff']:+.2f} (CI [{fa['lo']:.2f}, {fa['hi']:.2f}]), "
        f"final task {means.loc['dapr_weak', 'final_task'] - means.loc['lowest_margin', 'final_task']:+.2f}.")
if np.isfinite(JOINT_AVG):
    add(f"Joint training final average: {JOINT_AVG:.2f} (No replay {none_avg:.2f}). Share of that gap closed: "
        + '; '.join(f"{MLABEL[m]} {100 * (means.loc[m, 'final_average'] - none_avg) / (JOINT_AVG - none_avg):.0f}%"
                    for m in order_methods if m != 'none') + '.')
for m in [x for x in ['random', 'mfr', 'lowest_margin', 'dapr_weak'] if x in USED]:
    add(f"Transitions, {MLABEL[m]}: {t5.loc[m, 'forgotten_share']:.1f}% of pairs forgotten, "
        f"{t5.loc[m, 'recovered_share']:.1f}% recovered (net {t5.loc[m, 'net_change']:+.2f}); "
        f"forgetting rate {t5.loc[m, 'forgetting_rate']:.1f}% of known pairs, recovery rate {t5.loc[m, 'recovery_rate']:.1f}% of unknown pairs.")
if 'none' in q_rate.index:
    add(f"No replay: forgetting rate is {q_rate.loc['none', 1]:.1f}% for the weakest-learned quartile vs "
        f"{q_rate.loc['none', 4]:.1f}% for the strongest; mean margin drop {q_drop.loc['none', 1]:.3f} vs {q_drop.loc['none', 4]:.3f}.")
text = '\n'.join(f'- {l}' for l in lines)
print(text)
if SAVE:
    (OUT / 'key_numbers.txt').write_text(text + '\n')

## 15. Report map and export

| Report section | Use |
|---|---|
| §1 Introduction | Safety example (section 9 printout), "Without replay ..." sentence (key numbers) |
| §3.4 Experimental setup | Inventory table `T00_inventory`, Stage-1 check `T00_stage1_check` (one sentence) |
| §3.6 Metrics | Definitions in sections 2 and 7 of this notebook |
| §4.3 Results | **Table 1** `T1_main_results`, **Figure 1** `F1`, **Figure 2** `F2` (trade-off), Table 2a/2b statistics, Table 3 + `F4` per behavior, Table 4 final accuracy + joint |
| §4.3 Why Lowest margin wins | **Table 5 + `F5`** (forgotten vs recovered), Table 6 + `F7` (learned-margin quartiles) |
| §4.3 Mechanism | Table 7/7b/8 + `F9` (what was replayed) |
| §5.1 Progress / §5.2 Plan | Table 9 + `F10` (cost), Table 11 (selection rule) |
| Appendix | `F6` transition matrices, `F8` stage curves, `F11` + Tables 10a–c every cell |

In [ ]:
if SAVE:
    archive = shutil.make_archive(str(OUT.parent / 'midterm_outputs'), 'zip', OUT)
    print('Tables :', len(list((OUT / 'tables').glob('*.csv'))), 'CSV files')
    print('Figures:', len(list((OUT / 'figures').glob('*.png'))), 'PNG +', len(list((OUT / 'figures').glob('*.pdf'))), 'PDF')
    print('Zip    :', archive)
    print('Download it from your laptop with:')
    print(f'  scp <user>@discovery.usc.edu:{archive} ~/Downloads/')